In [14]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import struct
import matplotlib.pyplot as plt

In [40]:
# Read bout.bin, singles-like file containing hit information as seen by
# simulated detector, this is like what we would see in our real data.

# Final bout np array contains: 
# (ID of scintillator, event ID (used for grouping), energy in MeV)
file_path = "temp/bout.bin"
rows = []
i_scint = None
with open(file_path, "rb") as f:
    while True:
        marker = f.read(1)
        if not marker:
            break
        if marker == b"\xee":
            i_scint = struct.unpack("<i", f.read(4))[0]
        elif marker == b"\xff":
            event_id, time_ns, energy_mev = struct.unpack("<qdd", f.read(24))
            rows.append((i_scint, event_id, time_ns, energy_mev))
        else:
            raise ValueError(f"Unexpected marker {marker!r} at offset {f.tell()-1}")
bout = pd.DataFrame(rows, columns=["iScint", "eventId", "time_ns", "energy_MeV"])
bout.sort_values(by='eventId', inplace=True)
bout = np.array(bout)[:, [0, 1, 3]]
bout[:, 2] = bout[:, 2] * 1e6 # Convert energy to eV and save as int32 for simplicity
bout = bout.astype(np.int32) # 
print(bout.shape)
print(bout[0:5])

(2320531, 3)
[[    43      3 475602]
 [    27      3  35397]
 [    27      4 511000]
 [    17      5 358295]
 [    26      5  55795]]


In [47]:
# Various selection criteria on bout, comment as needed.

EBOUNDS = (0.190, 0.210) # The energy range of the lower hit of the ICS event

eventids = []
curreventid = bout[0, 1]
currenergies = []
for row in tqdm(bout):
    if not np.isclose(row[1], curreventid): # End of event, apply selection criteria
        # 1. Two hits per ICS event
        if len(currenergies) != 2:
            curreventid = row[1]
            currenergies = [row[2]]
            continue
        # 2. Energy sum is in photopeak range (using 0.45 MeV for now)
        if (currenergies[0] + currenergies[1]) < 0.450 * 1e6:
            curreventid = row[1]
            currenergies = [row[2]]
            continue
        currenergies.sort()
        # 3. Lower energy hit is in some specific range that we want to study
        if currenergies[0] > EBOUNDS[0] * 1e6 and currenergies[0] < EBOUNDS[1] * 1e6:
            eventids.append(curreventid)
        curreventid = row[1]
        currenergies = [row[2]]
    else:
        currenergies.append(row[2])
print(len(eventids))

  0%|          | 0/2320531 [00:00<?, ?it/s]

100%|██████████| 2320531/2320531 [00:20<00:00, 112898.81it/s]

2991


In [48]:
path = '/home/kale-chen/Documents/Geant4/TPPTsim/temp/gammaPath.txt'
kept = pd.DataFrame()
for chunk in tqdm(pd.read_csv(path, sep = ' ', chunksize=1000000)):
    kept = pd.concat([kept, chunk[chunk['eventId'].isin(eventids)]])
print(kept.shape)

18it [00:09,  2.00it/s]

(28063, 15)


In [50]:
# Individual event analysis here
for eventid in eventids:
    print(kept[kept['eventId'] == eventid])
    break

     eventId  trackId  parentId  step         process   preX_mm  preY_mm  \
441       61        1         0     1  Transportation  0.187457   2.3318   
442       61        1         0     2  Transportation  0.187457   2.3318   
443       61        1         0     3           compt  0.187457   2.3318   
444       61        1         0     4  Transportation  0.187457   2.3318   
445       61        1         0     5  Transportation  0.250667   3.1000   
446       61        1         0     6  Transportation  0.267124   3.3000   
447       61        1         0     7  Transportation  0.513973   6.3000   
448       61        1         0     8  Transportation  0.530429   6.5000   
449       61        1         0     9  Transportation  0.777279   9.5000   
450       61        1         0    10            phot  0.793735   9.7000   
451       61        6         1     1            phot  0.952996  11.6355   
452       61        5         1     1            phot  0.952996  11.6355   
453       61